# Lab 4: Your data, your MVP

The next class is your **MVP demo on Wednesday 21 October**. Today is the last lab before it, so today is about **your** project.

- **Layer A (30 minutes, everyone):** a warm-up on known data: coordinates that are not latitude and longitude, turned into ones your map can use, with GeoPandas.
- **Layer B (the working state):** your own data, through your own pipeline, on your own page.
- **Layer C:** your one interaction, or your reading-week plan.

Run the setup cell first.

## Setup: install GeoPandas (once)

This is the same as `%pip install pandas` in Lab 1. It can take a minute or two. If the next cell then says it cannot find geopandas, restart the kernel (the Restart button at the top of the notebook) and run it again. If the install itself fails, show your TA the last red line.

In [1]:
%pip install geopandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import geopandas as gpd
import json, os
print("GeoPandas", gpd.__version__, "is ready")

GeoPandas 1.2.0 is ready


## Layer A, Part 1. Numbers that are not latitude and longitude

This is the same sewer table, but the city has stored each pipe's position as **x and y in metres**, not as latitude and longitude. Many portals do this, including Montréal's.

**Predict before you run:** what will Leaflet do with a point at "latitude" 5452999 and "longitude" 486191?

In [3]:
df = pd.read_csv("../data/vancouver_sewers_utm10n.csv")
print(len(df), "rows")
df[["pipe_id", "flow_type", "x", "y"]].head()

39713 rows


,pipe_id,flow_type,x,y
0,2,Storm,486191.1,5452999.4
1,3,Storm,487132.9,5453738.0
2,4,Sanitary,490420.6,5451013.9
3,5,Storm,493089.7,5456538.1
4,6,Sanitary,490417.9,5450915.7


Those are metres east and north in a **projected coordinate system** called UTM zone 10N (its code is `EPSG:26910`). A projection flattens the curved Earth onto a grid, so distances come out in metres. Web maps want **degrees**: longitude and latitude in `EPSG:4326`.

You never convert these by hand, and you never swap or divide the numbers until the map "looks right". You tell GeoPandas what the numbers *are*, then ask it to convert them.

## Layer A, Part 2. Tell GeoPandas what the numbers are, then convert

Fill in the two blanks: the code of the system the data is in, and the code web maps use.

In [5]:
points = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["x"], df["y"]),
    crs="EPSG:32610",              # what the numbers ARE: UTM zone 10N
)
web = points.to_crs("EPSG:4326")  # what web maps want: longitude and latitude
web.geometry.head()

0    POINT (-123.18967 49.22962)
1    POINT (-123.17676 49.23628)
2    POINT (-123.13153 49.21184)
3    POINT (-123.09498 49.26157)
4    POINT (-123.13156 49.21096)
Name: geometry, dtype: geometry

**Check:** the points now read like `POINT (-123.19 49.23)`. GeoPandas writes longitude first, like GeoJSON.

## Layer A, Part 3. Cut it to the MVP, write, check

Two habits that keep a web map fast. **Filter to your question**: say this MVP is about combined sewers only, so keep only those rows. **Keep only what you show**: a GeoJSON file carries every column you leave in.

Then write it in **one line**, and compare that with the loop you wrote in Lab 3: GeoPandas does the same job, and now you know what it does inside. `COORDINATE_PRECISION=6` keeps six decimals, about 10 cm on the ground; without it GeoPandas writes fifteen, which is nanometres, and the file grows by a third for nothing.

In [7]:
combined = web[web["flow_type"] == "Combined"]       # keep one flow type: "Combined"
print(len(web), "pipes ->", len(combined), "combined")
keep = combined[["flow_type", "material", "length_m", "geometry"]]
keep.to_file("../data/sewers_from_utm.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)
print(round(os.path.getsize("../data/sewers_from_utm.geojson") / 1_000_000, 1), "MB")

39713 pipes -> 9656 combined
1.8 MB


### The check cell you will reuse for your own data

Change only the four settings at the top when you use it on your project. It reads the file back the way the browser will.

In [8]:
# ---- settings: change these for your own data ----
FILE = "../data/sewers_from_utm.geojson"
WEST, EAST, SOUTH, NORTH = -123.3, -122.9, 49.1, 49.4   # a box around your area, in degrees
EXPECTED = 9656           # how many features you expect (rows after your filters)
POPUP_FIELDS = ["flow_type", "material", "length_m"]
# ----------------------------------------------------

back = gpd.read_file(FILE)
problems = []
if len(back) != EXPECTED:
    problems.append(f"{len(back)} features, but you expected {EXPECTED}.")
if back.crs is None or back.crs.to_epsg() != 4326:
    problems.append(f"The file is in {back.crs}, not longitude and latitude (EPSG:4326).")
minx, miny, maxx, maxy = back.total_bounds
if not (WEST <= minx and maxx <= EAST and SOUTH <= miny and maxy <= NORTH):
    problems.append(f"Some features are outside your box: x {minx:.3f} to {maxx:.3f}, y {miny:.3f} to {maxy:.3f}.")
missing = [f for f in POPUP_FIELDS if f not in back.columns]
if missing:
    problems.append(f"Missing popup fields: {missing}.")
mb = os.path.getsize(FILE) / 1_000_000
if mb > 5:
    problems.append(f"The file is {mb:.1f} MB; over about 5 MB the page gets slow. Filter or simplify.")
print("Geometry types:", sorted(back.geom_type.unique()))
print("All checks passed" if not problems else "Not yet:\n- " + "\n- ".join(problems))

Geometry types: ['Point']
All checks passed


**Done with Layer A when** the check prints `All checks passed`. To see it, open `site/index.html` with Live Server: the combined sewers from Lab 3, now from metres. Then move to Layer B: that is today's working state.

---

# Layer B. Your data

Find your route in the handout's table, then use the matching recipe below **in a copy of this notebook inside your own project folder** (`notebooks/`). Each recipe ends by writing a GeoJSON file; then run the check cell above with your own settings.

Every recipe is commented out. Remove the `#` at the start of the lines you need (select them and press **Ctrl+/**), and change the names in capitals.

### Recipe 1. A CSV with latitude and longitude columns

In [ ]:
# df = pd.read_csv("../data/YOUR_FILE.csv")
# df = df.dropna(subset=["LAT_COLUMN", "LON_COLUMN"])          # count what this drops: print(len(df))
# gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["LON_COLUMN"], df["LAT_COLUMN"]), crs="EPSG:4326")
# gdf[["FIELD_1", "FIELD_2", "geometry"]].to_file("../data/YOUR_OUTPUT.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)

### Recipe 2. A CSV with x and y in metres (projected)

You need the system's code. Look in the portal's metadata for "projection", "CRS", "SCR" or "EPSG". Common ones: Montréal **MTM zone 8** is `EPSG:2950` (or the older `EPSG:32188`; the two differ by about a metre). Québec Lambert is `EPSG:32198`. Vancouver and much of BC: UTM zone 10N, `EPSG:26910`. Toronto: MTM zone 10, `EPSG:2952`. If you cannot find it, ask your TA before you guess.

In [ ]:
# df = pd.read_csv("../data/YOUR_FILE.csv")
# gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["X_COLUMN"], df["Y_COLUMN"]), crs="EPSG:YOUR_CODE")
# gdf = gdf.to_crs("EPSG:4326")
# gdf[["FIELD_1", "FIELD_2", "geometry"]].to_file("../data/YOUR_OUTPUT.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)

### Recipe 3. A shapefile, GeoPackage or GeoJSON (points, lines or areas)

A shapefile is several files with the same name (`.shp`, `.shx`, `.dbf`, `.prj` and more); keep them together in `data/` and point at the `.shp`. GeoPandas reads the coordinate system from the `.prj`. Lines and areas are often far more detailed than a web map needs: `simplify` removes points closer together than the tolerance, **in the file's own units**, so do it before converting to degrees.

In [ ]:
# gdf = gpd.read_file("../data/YOUR_FILE.shp")      # or .gpkg, or .geojson
# print(gdf.crs, len(gdf), sorted(gdf.geom_type.unique()))
# print(gdf.columns.tolist())
# gdf = gdf[["FIELD_1", "FIELD_2", "geometry"]]
# # Lines or areas only: simplify to about 5 metres if the file is in metres (skip if gdf.crs says degrees)
# # gdf["geometry"] = gdf.geometry.simplify(5)
# gdf = gdf.to_crs("EPSG:4326")
# gdf.to_file("../data/YOUR_OUTPUT.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)

### Recipe 4. Too big: filter first

Your MVP needs one borough, one year or one category, not the whole province. Filter on a column, or clip to a box. Count before and after, and put the numbers in your README.

In [ ]:
# print("before:", len(gdf))
# gdf = gdf[gdf["BOROUGH_COLUMN"] == "Plateau-Mont-Royal"]                 # filter by a value
# # or by year:   gdf = gdf[pd.to_datetime(gdf["DATE_COLUMN"]).dt.year == 2025]
# # or by a box (in degrees, after to_crs 4326):   gdf = gdf.cx[-73.62:-73.55, 45.50:45.55]
# print("after:", len(gdf))

### Route 5. Addresses only, or no data yet

Turning addresses into coordinates (geocoding) at scale needs a service with usage limits and terms. For an MVP: look for a version of the same data with coordinates (portals often publish both), or, for under about 50 places, add `lat` and `lon` columns yourself: on openstreetmap.org, right-click each place, choose "Centre map here", and read the two numbers after the zoom in the address bar (`#map=18/LATITUDE/LONGITUDE`). Then use Recipe 1. Say so in your README. No dataset yet: tell your TA now; choosing one is today's working state.